In [2]:
from quantrisk import ARMAGARCH, Backtest, Plotter, Risk, Statistics
import numpy as np
import pandas as pd

For this demo we will be analyzing log returns of Apple (AAPL) from 01/02/2015-05/29/2026. We will use the last 1000 trading days in our demo.

In [7]:
path = "DATA/AAPL.csv"
df = pd.read_csv(path, header=[0, 1], index_col=0, parse_dates=True)
prices = df["Close"].dropna()
returns = np.log(prices / prices.shift(1)).dropna()
df["Returns"] = returns
data = df["Returns"].iloc[-1000:]

# 1. ARMA(1,1)-GARCH(1,1)

We will first fit an ARMA(1,1)-GARCH(1,1) model with Gaussian shocks to our returns data. Later (see Sec. 2) we will see how we can model the data using shocks from other distributions. We use the following notation:
$$r_t = \mathbb{E}[r_{t}|\mathcal{F}_{t-1}] + \varepsilon_t = \xi + \phi r_{t-1} + \theta \varepsilon_{t-1} + \varepsilon_{t}$$
$$\sigma_{t}^2 = C + B \sigma_{t-1}^2 + A\varepsilon_{t-1}^2$$
$$\varepsilon_t = \sigma_t z_t$$
$$z_t \sim N(0,1)$$
Here $\mathcal{F}_t$ denotes the filtration at time $t$. we refer to $\{e_t\}$ as the innovations and $\{z_t\}$ as the shocks. 

First create an ARMAGARCH object and fit the model.

In [9]:
AAPL_AG = ARMAGARCH()
AAPL_AG.fit(data)

The fitted ARMA and GARCH parameters are stored in a dictionary as object attributes:

In [18]:
print("Fitted ARMA parameters:")
for k,v in AAPL_AG.ARMA_params.items():
    print(f"{k} = {v}")

Fitted ARMA parameters:
phi = -0.12531441181056213
theta = 0.1735936554506209
xi = 0.0008376853332238213


In [19]:
print("Fitted GARCH parameters:")
for k,v in AAPL_AG.GARCH_params.items():
    print(f"{k} = {v}")

Fitted GARCH parameters:
A = 0.07036095548848638
B = 0.8927952043244026
C = 1.0765482484116214e-05
